### Project Setup and Library Imports

First, we'll import the necessary Python libraries for data manipulation, visualization, and numerical operations. These libraries will be used throughout our LSTM project for data loading, preprocessing, and model development.

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### Load the Dataset

Next, we'll load our dataset, which contains quotes and their authors, into a Pandas DataFrame. The dataset is stored in a CSV file named `qoute_dataset.csv`.

In [13]:
df = pd.read_csv("qoute_dataset.csv")

### Initial Data Inspection

To get a quick overview of our data, we'll display the first few rows of the DataFrame. This helps us understand the structure and content of our dataset.

In [14]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


### Examine a Single Data Point

Let's look at a specific quote from the dataset to understand its format before we start cleaning.

In [15]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

### Check Dataset Dimensions

We'll check the number of rows and columns in our DataFrame to understand the size of our dataset.

In [16]:
df.shape

(3038, 2)

### Extracting the Target Column

For our LSTM model, we are primarily interested in the 'quote' column as it contains the text data we will use for next word prediction. We will extract this column into a new Series.

In [17]:
quotes = df["quote"]
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


### Convert Text to Lowercase

To ensure consistency and reduce vocabulary size, we convert all the text in the 'quotes' series to lowercase. This helps treat words like 'The' and 'the' as the same token.

In [18]:
quotes=quotes.str.lower()

### Verify Lowercasing

After converting to lowercase, we'll display the first few quotes again to confirm the transformation.

In [19]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,"“it is our choices, harry, that show what we t..."
2,“there are only two ways to live your life. on...
3,"“the person, be it gentleman or lady, who has ..."
4,"“imperfection is beauty, madness is genius and..."


### Remove Punctuation

Punctuation marks often don't contribute to the meaning of words in a next-word prediction task and can increase vocabulary complexity. We'll remove them from the quotes using `str.maketrans` and `apply`.

In [20]:
import string
transator =  str.maketrans('','',string.punctuation)
quotes = quotes.apply(lambda x: x.translate(transator))

### Verify Punctuation Removal

Finally, we'll check the first few quotes to see the effect of punctuation removal.

In [21]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


### Import Tokenizer

We import the `Tokenizer` class from `tensorflow.keras.preprocessing.text`. This tool is essential for converting text into sequences of numbers, which is a required step before feeding text data into neural networks like LSTMs.

In [22]:
from tensorflow.keras.preprocessing.text import Tokenizer

### Initialize and Fit Tokenizer

We define a `vocab_size` (maximum number of words to keep, based on word frequency) and initialize the `Tokenizer`. We then fit the tokenizer on our cleaned `quotes` data. This step builds the vocabulary and assigns unique integer IDs to each word.

In [23]:
vocab_size = 10000

tokinizer = Tokenizer(num_words=vocab_size)
tokinizer.fit_on_texts(quotes)

### Inspect Word Index

After fitting the tokenizer, we can inspect the `word_index` attribute, which is a dictionary mapping words to their integer IDs. We'll print the total number of unique words found and the first 10 entries to verify the tokenization process.

In [24]:
word_index=tokinizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

### Convert Text to Sequences

Now, we convert our text `quotes` into sequences of integers using the fitted tokenizer. Each quote becomes a list of numbers, where each number corresponds to a word in our vocabulary.

In [25]:
sequence = tokinizer.texts_to_sequences(quotes)

### Display Original Quotes

To understand the data conversion, we'll display the first three original (cleaned) quotes.

In [26]:
for i in range(3):
  print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


### Display Tokenized Sequences

Next, we'll display the corresponding tokenized integer sequences for the first three quotes. This shows how each word has been mapped to a numerical ID.

In [27]:
for i in range(3):
  print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


### Create Input-Output Pairs for Next Word Prediction

For a next-word prediction model, we need to create input-output pairs. Each sequence is broken down into multiple sub-sequences. For a sequence `[w1, w2, w3, w4]`, the pairs would be:
- Input: `[w1]`, Output: `w2`
- Input: `[w1, w2]`, Output: `w3`
- Input: `[w1, w2, w3]`, Output: `w4`

This process generates `X` (input sequences) and `y` (target next words).

In [28]:
X=[]
y=[]

for seq in sequence:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)

### Check Number of Input Sequences

We'll check the total number of input sequences (`X`) generated after splitting all quotes into input-output pairs. This gives us an idea of the size of our training data.

In [29]:
len(X)

85271

### Check Number of Output Labels

Similarly, we'll check the total number of output labels (`y`). This should match the number of input sequences, as each input sequence corresponds to one target word.

In [30]:
len(y)

85271

### Determine Maximum Sequence Length

Neural networks often require fixed-size inputs. We need to find the maximum length among all input sequences in `X`. This value will be used for padding shorter sequences to ensure uniform input size for the model.

### Pad Input Sequences

To ensure all input sequences have the same length, which is required for neural network input, we'll pad the `X` sequences. We'll use pre-padding, adding zeros to the beginning of shorter sequences, up to the `max_len` determined earlier.

In [31]:
max_len=max(len(i) for i in X)
print(max_len)

745


In [32]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X,maxlen=max_len,padding='pre')

### Convert Target Labels to NumPy Array

We convert the `y` (target words) list into a NumPy array. This is a standard practice for preparing data for TensorFlow/Keras models, which often expect NumPy arrays as input.

In [33]:
y=np.array(y)

### Check Padded Input Shape

We'll check the shape of the `X_padded` array to confirm that all sequences now have the uniform `max_len` and to see the total number of samples.

In [34]:
X_padded.shape

(85271, 745)

### One-Hot Encode Target Labels

For multi-class classification (predicting the next word from a vocabulary), the target labels (`y`) need to be one-hot encoded. This converts each integer word ID into a binary vector of size `vocab_size`.

In [35]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y,num_classes=vocab_size)

### Check Original Target Labels Shape

We'll check the shape of the original `y` array before one-hot encoding to compare it with the one-hot encoded version and understand the transformation.

In [36]:
y.shape

(85271,)

### Check One-Hot Encoded Target Labels Shape

After one-hot encoding, we'll examine the shape of `y_one_hot`. This will show the number of samples and the size of the one-hot vector, which should be equal to our `vocab_size`.

In [37]:
y_one_hot.shape

(85271, 10000)

### Import Keras Model and Layers

We'll import the necessary components from TensorFlow Keras to define our neural network model. This includes `Sequential` for building the model layer by layer, and layers like `Embedding`, `LSTM`, and `Dense`.

In [38]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,SimpleRNN

### Define Model Hyperparameters

Before building the model, we define key hyperparameters such as `embedding_dim` (the dimensionality of the word embedding vectors) and `rnn_units` (the number of units in the LSTM layer). These values will influence the model's capacity and performance.

In [39]:
embedding_dim = 50
rnn_units = 128

In [40]:
rnn_model =Sequential()

rnn_model.add(
Embedding(input_dim=vocab_size , output_dim= embedding_dim,input_length=max_len))

rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size,activation='softmax'))



/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [41]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [42]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [43]:
lstm_model =Sequential()
lstm_model.add(Embedding(input_dim=vocab_size , output_dim =embedding_dim,
                         input_length=max_len)
)
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size,activation='softmax'))

In [44]:
lstm_model.compile(
    optimizer = 'adam',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)

In [45]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [46]:
epochs = 10
batch_size = 128


In [47]:
history_rnn = rnn_model.fit(
    X_padded,
    y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1
)

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 46s 69ms/step - accuracy: 0.0459 - loss: 6.6944 - val_accuracy: 0.0572 - val_loss: 6.5172
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - accuracy: 0.0754 - loss: 6.1004 - val_accuracy: 0.0894 - val_loss: 6.3298
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.1011 - loss: 5.7347 - val_accuracy: 0.1015 - val_loss: 6.2829
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - accuracy: 0.1184 - loss: 5.4324 - val_accuracy: 0.1034 - val_loss: 6.3440
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - accuracy: 0.1330 - loss: 5.1711 - val_accuracy: 0.1093 - val_loss: 6.3941
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.1447 - loss: 4.9352 - val_accuracy: 0.1075 - val_loss: 6.4322
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 63ms/step - accuracy: 0.1594 - loss: 4.7149 - val_accuracy: 0.1075 - val_loss: 6.5062
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - accuracy: 0.1756 - loss: 4.5111 - 

In [ ]:
from tensorflow.keras.models import load_model

lstm_model = load_model("lstm_model.h5")


In [ ]:
lstm_model.save()